# Корреляционный анализ

**Шаг 3 плана.** Корреляционная матрица фичей, выявление сильно коррелированных пар, корреляция фичей с финальным target `tp_sl_1_05` (колонка `target`).

## 1. Импорты и загрузка

In [1]:
import sys
import os
import numpy as np
import pandas as pd

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.feature_pipeline import get_feature_columns

path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
if not os.path.exists(path):
    raise FileNotFoundError(f'Запустите 04_Data_Labeling_And_Feature_Loading.ipynb. Файл не найден: {path}')

df = pd.read_parquet(path)
print(f'Загружено: {len(df):,} строк, {df["session_key"].nunique()} сессий')

Загружено: 344,415 строк, 1048 сессий


## 2. Фичи

In [2]:
# Базовый набор фичей + режимные признаки
feat_base = [c for c in get_feature_columns(include_symbol=True)]
extra = ['rd_regime', 'rd_regime_transition']
feat_cols = [c for c in feat_base if c in df.columns] + [c for c in extra if c in df.columns and c not in feat_base]
print(f'Фичей: {len(feat_cols)}')
print(feat_cols)

Фичей: 27
['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'bb_width', 'bb_position', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'symbol_encoded', 'rd_regime', 'rd_regime_transition']


## 3. Корреляционная матрица

In [3]:
valid = df[feat_cols].dropna()
corr_matrix = valid.corr()
print('Корреляционная матрица (размер:', corr_matrix.shape[0], 'x', corr_matrix.shape[1], ')')

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, vmin=-0.8, vmax=0.8, ax=ax, annot=False)
    plt.title('Корреляционная матрица фичей')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib/seaborn не установлены — пропуск heatmap')

Корреляционная матрица (размер: 27 x 27 )


matplotlib/seaborn не установлены — пропуск heatmap


## 4. Сильно коррелированные пары (|r| > 0.8)

In [4]:
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], r))

print('Сильно коррелированные пары (|r| > 0.8):')
if high_corr_pairs:
    for a, b, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
        print(f'  {a} <-> {b}: r = {r:.4f}')
else:
    print('  (нет пар с |r| > 0.8)')

Сильно коррелированные пары (|r| > 0.8):
  macd_line <-> macd_signal: r = 0.9508
  bb_width <-> volatility_14: r = 0.8412
  rsi_14 <-> bb_position: r = 0.8347
  rd_zscore_30 <-> bb_position: r = 0.8178


## 5. Корреляция фичей с target (BUY/SELL, без HOLD)

In [5]:
TARGET_COL = 'target'
binary = df.dropna(subset=[TARGET_COL] + feat_cols)
binary = binary[binary[TARGET_COL].isin([1.0, -1.0])]

target_corrs = []
for c in feat_cols:
    r = binary[c].corr(binary[TARGET_COL])
    target_corrs.append({'feature': c, 'corr': r, 'abs_corr': abs(r)})

corr_df = pd.DataFrame(target_corrs).sort_values('abs_corr', ascending=False)
print('Топ-10 фичей по |corr| с target:')
print(corr_df.head(10).to_string(index=False))

# Сохраняем для 06_Feature_Selection
out_path = os.path.join(_root, 'outputs', 'correlation_with_target_tp_sl_1_05.csv')
os.makedirs(os.path.join(_root, 'outputs'), exist_ok=True)
corr_df.to_csv(out_path, index=False)
print(f'\nСохранено: {out_path}')

Топ-10 фичей по |corr| с target:

        feature      corr  abs_corr
   rd_zscore_30  0.099284  0.099284
      rd_regime  0.084567  0.084567
       rd_mom_1  0.070880  0.070880
       rd_mom_5  0.067506  0.067506
      rd_mom_10  0.056203  0.056203
rd_acceleration  0.022783  0.022783
 close_position -0.016732  0.016732
          ret_1 -0.015715  0.015715
      rd_ema_20  0.013075  0.013075
       bb_width -0.012029  0.012029

Сохранено: c:\project\trading_bot_2Engine\outputs\correlation_with_target_tp_sl_1_05.csv


## 6. Сохранение исключений (отдельно)

Исключения фиксируем после полного анализа 27 фичей, чтобы можно было легко вернуть признаки или расширить набор в будущем.

In [6]:
# Ручные исключения по итогам корреляционного анализа.
# Этот список читается в 06_Feature_Selection и вычитается из feat_cols
# до автоматического отбора (шум, мультиколлинеарность).
# Итоговый FEATURES_SELECTED сохраняется в 06 — единственный источник правды
# для 07, обучения и API.
EXCLUDED_FEATURES = ['is_weekend', 'bb_position', 'bb_width']

out_dir = os.path.join(_root, 'outputs')
os.makedirs(out_dir, exist_ok=True)

excluded_path = os.path.join(out_dir, 'excluded_features_tp_sl_1_05.txt')
with open(excluded_path, 'w', encoding='utf-8') as f:
    for c in EXCLUDED_FEATURES:
        f.write(c + '\n')
print(f'Сохранён список исключений ({len(EXCLUDED_FEATURES)} фичи): {excluded_path}')
print('Исключены:', EXCLUDED_FEATURES)

Сохранён список исключений (3 фичи): c:\project\trading_bot_2Engine\outputs\excluded_features_tp_sl_1_05.txt
Исключены: ['is_weekend', 'bb_position', 'bb_width']


## Итог шага 3

- Корреляционная матрица и корреляция с target считаются по **полному набору (27 фичей)**.
- Выявлены сильно коррелированные пары (|r| > 0.8).
- Топ-10 фичей по связи с target → `outputs/correlation_with_target_tp_sl_1_05.csv`.
- Ручные исключения (`is_weekend`, `bb_position`, `bb_width`) → `outputs/excluded_features_tp_sl_1_05.txt`.
- Исключения применяются в **06_Feature_Selection** (не здесь), где формируется единый `features_selected_tp_sl_1_05.txt`.